In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)

In [5]:
data = pd.read_csv("../data/visitor_prediction_features_snapshot.csv")
data["month"] = pd.to_datetime(data["month"])

data = data.sort_values(["country", "month"]).reset_index(drop=True)
data.head()

,country,month,visitor_arrivals,gdp,exchange_rate,public_holiday_count,aircraft_passengers,traffic_volume,hotel_rate,hotel_occupancy
0,Australia,1978-01-01,20379.0,135608.0,NaN,2,NaN,NaN,NaN,NaN
1,Australia,1978-02-01,18852.0,135608.0,NaN,0,NaN,NaN,NaN,NaN
2,Australia,1978-03-01,20819.0,135608.0,NaN,9,NaN,NaN,NaN,NaN
3,Australia,1978-04-01,18697.0,135608.0,NaN,1,NaN,NaN,NaN,NaN
4,Australia,1978-05-01,19797.0,135608.0,NaN,3,NaN,NaN,NaN,NaN


In [6]:
total_data = pd.read_csv("../data/total_visitor_arrivals_monthly.csv")
total_data["month"] = pd.to_datetime(total_data["month"])
print(total_data.shape)
total_data.head()

(569, 2)


,month,total_visitor_arrivals
0,1978-01-01,165678.0
1,1978-02-01,163401.0
2,1978-03-01,166743.0
3,1978-04-01,167155.0
4,1978-05-01,169163.0


In [7]:
print("Columns in dataset:")
for col in data.columns:
    print(col)

Columns in dataset:
country
month
visitor_arrivals
gdp
exchange_rate
public_holiday_count
aircraft_passengers
traffic_volume
hotel_rate
hotel_occupancy


In [9]:
data["log_visitor_arrivals"] = np.log1p(data["visitor_arrivals"])
data[["country", "month", "visitor_arrivals", "log_visitor_arrivals"]].head()

,country,month,visitor_arrivals,log_visitor_arrivals
0,Australia,1978-01-01,20379.0,9.922309
1,Australia,1978-02-01,18852.0,9.844427
2,Australia,1978-03-01,20819.0,9.943669
3,Australia,1978-04-01,18697.0,9.836172
4,Australia,1978-05-01,19797.0,9.893336


In [11]:
total_data["log_visitor_arrivals"] = np.log1p(total_data["total_visitor_arrivals"])
total_data[["month", "total_visitor_arrivals", "log_visitor_arrivals"]].head()

,month,total_visitor_arrivals,log_visitor_arrivals
0,1978-01-01,165678.0,12.017807
1,1978-02-01,163401.0,12.003969
2,1978-03-01,166743.0,12.024215
3,1978-04-01,167155.0,12.026683
4,1978-05-01,169163.0,12.038624


In [12]:
data["year"] = data["month"].dt.year
data["month_num"] = data["month"].dt.month
data["quarter"] = data["month"].dt.quarter

data["month_sin"] = np.sin(2 * np.pi * data["month_num"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month_num"] / 12)

data[["country", "month", "year", "month_num", "quarter", "month_sin", "month_cos"]].head()

,country,month,year,month_num,quarter,month_sin,month_cos
0,Australia,1978-01-01,1978,1,1,0.500000,8.660254e-01
1,Australia,1978-02-01,1978,2,1,0.866025,5.000000e-01
2,Australia,1978-03-01,1978,3,1,1.000000,6.123234e-17
3,Australia,1978-04-01,1978,4,2,0.866025,-5.000000e-01
4,Australia,1978-05-01,1978,5,2,0.500000,-8.660254e-01


In [13]:
total_data["year"] = total_data["month"].dt.year
total_data["month_num"] = total_data["month"].dt.month
total_data["quarter"] = total_data["month"].dt.quarter

total_data["month_sin"] = np.sin(2 * np.pi * total_data["month_num"] / 12)
total_data["month_cos"] = np.cos(2 * np.pi * total_data["month_num"] / 12)

total_data[["month", "year", "month_num", "quarter", "month_sin", "month_cos"]].head()

,month,year,month_num,quarter,month_sin,month_cos
0,1978-01-01,1978,1,1,0.500000,8.660254e-01
1,1978-02-01,1978,2,1,0.866025,5.000000e-01
2,1978-03-01,1978,3,1,1.000000,6.123234e-17
3,1978-04-01,1978,4,2,0.866025,-5.000000e-01
4,1978-05-01,1978,5,2,0.500000,-8.660254e-01


In [15]:
for lag in [1, 2, 3, 6, 12]:
    data[f"log_arrivals_lag_{lag}"] = data.groupby("country")["log_visitor_arrivals"].shift(lag)

lag_cols = [col for col in data.columns if col.startswith("log_arrivals_lag_")]
data[["country", "month"] + lag_cols].head(15)

,country,month,log_arrivals_lag_1,log_arrivals_lag_2,log_arrivals_lag_3,log_arrivals_lag_6,log_arrivals_lag_12
0,Australia,1978-01-01,NaN,NaN,NaN,NaN,NaN
1,Australia,1978-02-01,9.922309,NaN,NaN,NaN,NaN
2,Australia,1978-03-01,9.844427,9.922309,NaN,NaN,NaN
3,Australia,1978-04-01,9.943669,9.844427,9.922309,NaN,NaN
4,Australia,1978-05-01,9.836172,9.943669,9.844427,NaN,NaN
5,Australia,1978-06-01,9.893336,9.836172,9.943669,NaN,NaN
6,Australia,1978-07-01,9.829357,9.893336,9.836172,9.922309,NaN
7,Australia,1978-08-01,9.801178,9.829357,9.893336,9.844427,NaN
8,Australia,1978-09-01,9.858909,9.801178,9.829357,9.943669,NaN
9,Australia,1978-10-01,9.830271,9.858909,9.801178,9.836172,NaN


In [16]:
for lag in [1, 2, 3, 6, 12]:
    total_data[f"log_arrivals_lag_{lag}"] = total_data["log_visitor_arrivals"].shift(lag)

lag_cols = [col for col in total_data.columns if col.startswith("log_arrivals_lag_")]
total_data[["month"] + lag_cols].head(15)

,month,log_arrivals_lag_1,log_arrivals_lag_2,log_arrivals_lag_3,log_arrivals_lag_6,log_arrivals_lag_12
0,1978-01-01,NaN,NaN,NaN,NaN,NaN
1,1978-02-01,12.017807,NaN,NaN,NaN,NaN
2,1978-03-01,12.003969,12.017807,NaN,NaN,NaN
3,1978-04-01,12.024215,12.003969,12.017807,NaN,NaN
4,1978-05-01,12.026683,12.024215,12.003969,NaN,NaN
5,1978-06-01,12.038624,12.026683,12.024215,NaN,NaN
6,1978-07-01,11.980764,12.038624,12.026683,12.017807,NaN
7,1978-08-01,12.011747,11.980764,12.038624,12.003969,NaN
8,1978-09-01,12.106357,12.011747,11.980764,12.024215,NaN
9,1978-10-01,12.070007,12.106357,12.011747,12.026683,NaN


In [18]:
grouped_log = data.groupby("country")["log_visitor_arrivals"]

data["log_arrivals_rollmean_3"] = grouped_log.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
data["log_arrivals_rollmean_6"] = grouped_log.shift(1).rolling(6).mean().reset_index(level=0, drop=True)
data["log_arrivals_rollmean_12"] = grouped_log.shift(1).rolling(12).mean().reset_index(level=0, drop=True)

data["log_arrivals_rollstd_3"] = grouped_log.shift(1).rolling(3).std().reset_index(level=0, drop=True)
data["log_arrivals_rollstd_6"] = grouped_log.shift(1).rolling(6).std().reset_index(level=0, drop=True)

roll_cols = [col for col in data.columns if "roll" in col]
data[["country", "month"] + roll_cols].head(15)

,country,month,log_arrivals_rollmean_3,log_arrivals_rollmean_6,log_arrivals_rollmean_12,log_arrivals_rollstd_3,log_arrivals_rollstd_6
0,Australia,1978-01-01,NaN,NaN,NaN,NaN,NaN
1,Australia,1978-02-01,NaN,NaN,NaN,NaN,NaN
2,Australia,1978-03-01,NaN,NaN,NaN,NaN,NaN
3,Australia,1978-04-01,9.903469,NaN,NaN,0.052235,NaN
4,Australia,1978-05-01,9.874756,NaN,NaN,0.059823,NaN
5,Australia,1978-06-01,9.891059,NaN,NaN,0.053785,NaN
6,Australia,1978-07-01,9.852955,9.878212,NaN,0.035137,0.048484
7,Australia,1978-08-01,9.841290,9.858023,NaN,0.047224,0.051571
8,Australia,1978-09-01,9.829814,9.860437,NaN,0.028868,0.051144
9,Australia,1978-10-01,9.830119,9.841537,NaN,0.028866,0.031362


In [20]:
total_data["log_arrivals_rollmean_3"] = total_data["log_visitor_arrivals"].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
total_data["log_arrivals_rollmean_6"] = total_data["log_visitor_arrivals"].shift(1).rolling(6).mean().reset_index(level=0, drop=True)
total_data["log_arrivals_rollmean_12"] = total_data["log_visitor_arrivals"].shift(1).rolling(12).mean().reset_index(level=0, drop=True)

total_data["log_arrivals_rollstd_3"] = total_data["log_visitor_arrivals"].shift(1).rolling(3).std().reset_index(level=0, drop=True)
total_data["log_arrivals_rollstd_6"] = total_data["log_visitor_arrivals"].shift(1).rolling(6).std().reset_index(level=0, drop=True)

roll_cols = [col for col in total_data.columns if "roll" in col]
total_data[["month"] + roll_cols].head(15)

,month,log_arrivals_rollmean_3,log_arrivals_rollmean_6,log_arrivals_rollmean_12,log_arrivals_rollstd_3,log_arrivals_rollstd_6
0,1978-01-01,NaN,NaN,NaN,NaN,NaN
1,1978-02-01,NaN,NaN,NaN,NaN,NaN
2,1978-03-01,NaN,NaN,NaN,NaN,NaN
3,1978-04-01,12.015330,NaN,NaN,0.010348,NaN
4,1978-05-01,12.018289,NaN,NaN,0.012463,NaN
5,1978-06-01,12.029841,NaN,NaN,0.007706,NaN
6,1978-07-01,12.015357,12.015344,NaN,0.030547,0.020398
7,1978-08-01,12.010379,12.014334,NaN,0.028954,0.020402
8,1978-09-01,12.032956,12.031398,NaN,0.065428,0.041701
9,1978-10-01,12.062704,12.039030,NaN,0.047726,0.044237


In [21]:
data["log_arrivals_diff_1"] = data.groupby("country")["log_visitor_arrivals"].diff(1)
data["log_arrivals_diff_12"] = data.groupby("country")["log_visitor_arrivals"].diff(12)

data["arrivals_pct_change_1"] = data.groupby("country")["visitor_arrivals"].pct_change(1)
data["arrivals_pct_change_12"] = data.groupby("country")["visitor_arrivals"].pct_change(12)

change_cols = [
    "log_arrivals_diff_1",
    "log_arrivals_diff_12",
    "arrivals_pct_change_1",
    "arrivals_pct_change_12"
]
data[["country", "month"] + change_cols].head(15)

,country,month,log_arrivals_diff_1,log_arrivals_diff_12,arrivals_pct_change_1,arrivals_pct_change_12
0,Australia,1978-01-01,NaN,NaN,NaN,NaN
1,Australia,1978-02-01,-0.077882,NaN,-0.074930,NaN
2,Australia,1978-03-01,0.099242,NaN,0.104339,NaN
3,Australia,1978-04-01,-0.107497,NaN,-0.101926,NaN
4,Australia,1978-05-01,0.057164,NaN,0.058833,NaN
5,Australia,1978-06-01,-0.063980,NaN,-0.061979,NaN
6,Australia,1978-07-01,-0.028179,NaN,-0.027787,NaN
7,Australia,1978-08-01,0.057731,NaN,0.059433,NaN
8,Australia,1978-09-01,-0.028637,NaN,-0.028232,NaN
9,Australia,1978-10-01,-0.021864,NaN,-0.021628,NaN


In [24]:
total_data["log_arrivals_diff_1"] = total_data["log_visitor_arrivals"].diff(1)
total_data["log_arrivals_diff_12"] = total_data["log_visitor_arrivals"].diff(12)

total_data["arrivals_pct_change_1"] = total_data["total_visitor_arrivals"].pct_change(1)
total_data["arrivals_pct_change_12"] = total_data["total_visitor_arrivals"].pct_change(12)

change_cols = [
    "log_arrivals_diff_1",
    "log_arrivals_diff_12",
    "arrivals_pct_change_1",
    "arrivals_pct_change_12"
]
total_data[["month"] + change_cols].head(15)

,month,log_arrivals_diff_1,log_arrivals_diff_12,arrivals_pct_change_1,arrivals_pct_change_12
0,1978-01-01,NaN,NaN,NaN,NaN
1,1978-02-01,-0.013839,NaN,-0.013744,NaN
2,1978-03-01,0.020246,NaN,0.020453,NaN
3,1978-04-01,0.002468,NaN,0.002471,NaN
4,1978-05-01,0.011941,NaN,0.012013,NaN
5,1978-06-01,-0.057860,NaN,-0.056218,NaN
6,1978-07-01,0.030983,NaN,0.031468,NaN
7,1978-08-01,0.094610,NaN,0.099231,NaN
8,1978-09-01,-0.036351,NaN,-0.035698,NaN
9,1978-10-01,0.012202,NaN,0.012277,NaN


In [26]:
external_candidates = [
    "gdp",
    "exchange_rate",
    "aircraft_passengers",
    "traffic_volume",
    "hotel_rate",
    "holiday_count"
]

available_external = [col for col in external_candidates if col in data.columns]
print("Available external columns:", available_external)

Available external columns: ['gdp', 'exchange_rate', 'aircraft_passengers', 'traffic_volume', 'hotel_rate']


In [27]:
for col in available_external:
    grp = data.groupby("country")[col]
    data[f"{col}_lag_1"] = grp.shift(1)
    data[f"{col}_lag_3"] = grp.shift(3)
    data[f"{col}_rollmean_3"] = grp.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
    data[f"{col}_rollmean_6"] = grp.shift(1).rolling(6).mean().reset_index(level=0, drop=True)

new_external_cols = [c for c in data.columns if any(
    c == base or c.startswith(f"{base}_") for base in available_external
)]
data[["country", "month"] + new_external_cols[:20]].tail(12)

,country,month,gdp,exchange_rate,aircraft_passengers,traffic_volume,hotel_rate,gdp_lag_1,gdp_lag_3,gdp_rollmean_3,...,exchange_rate_lag_3,exchange_rate_rollmean_3,exchange_rate_rollmean_6,aircraft_passengers_lag_1,aircraft_passengers_lag_3,aircraft_passengers_rollmean_3,aircraft_passengers_rollmean_6,traffic_volume_lag_1,traffic_volume_lag_3,traffic_volume_rollmean_3
9092,United States,2024-06-01,29184900.0,1.351589,5621575.0,NaN,260.2,29184900.0,29184900.0,2.918490e+07,...,1.340491,1.349253,1.343159,5482723.0,5726451.0,5.537361e+06,5.534737e+06,NaN,NaN,NaN
9093,United States,2024-07-01,29184900.0,1.346810,5704270.0,NaN,269.2,29184900.0,29184900.0,2.918490e+07,...,1.355940,1.352952,1.346386,5621575.0,5402910.0,5.502403e+06,5.502562e+06,NaN,NaN,NaN
9094,United States,2024-08-01,29184900.0,1.316448,5729606.0,NaN,273.8,29184900.0,29184900.0,2.918490e+07,...,1.351328,1.349909,1.348443,5704270.0,5482723.0,5.602856e+06,5.548435e+06,NaN,NaN,NaN
9095,United States,2024-09-01,29184900.0,1.296952,5399080.0,NaN,312.7,29184900.0,29184900.0,2.918490e+07,...,1.351589,1.338282,1.343767,5729606.0,5621575.0,5.685150e+06,5.611256e+06,NaN,NaN,NaN
9096,United States,2024-10-01,29184900.0,1.309387,5650043.0,NaN,270.4,29184900.0,29184900.0,2.918490e+07,...,1.346810,1.320070,1.336511,5399080.0,5704270.0,5.610985e+06,5.556694e+06,NaN,NaN,NaN
9097,United States,2024-11-01,29184900.0,1.335423,5741007.0,NaN,267.6,29184900.0,29184900.0,2.918490e+07,...,1.316448,1.307595,1.328752,5650043.0,5729606.0,5.592910e+06,5.597883e+06,NaN,NaN,NaN
9098,United States,2024-12-01,29184900.0,1.350140,6410825.0,NaN,275.3,29184900.0,29184900.0,2.918490e+07,...,1.296952,1.313920,1.326101,5741007.0,5399080.0,5.596710e+06,5.640930e+06,NaN,NaN,NaN
9099,United States,2025-01-01,30507217.0,1.361214,6155064.0,NaN,272.6,29184900.0,29184900.0,2.918490e+07,...,1.309387,1.331650,1.325860,6410825.0,5650043.0,5.933958e+06,5.772472e+06,NaN,NaN,NaN
9100,United States,2025-02-01,30507217.0,1.346776,5441857.0,NaN,274.6,30507217.0,29184900.0,2.962567e+07,...,1.335423,1.348926,1.328261,6155064.0,5741007.0,6.102299e+06,5.847604e+06,NaN,NaN,NaN
9101,United States,2025-03-01,30507217.0,1.335761,5616764.0,NaN,264.7,30507217.0,29184900.0,3.006644e+07,...,1.350140,1.352710,1.333315,5441857.0,6410825.0,6.002582e+06,5.799646e+06,NaN,NaN,NaN


In [29]:
if "holiday_count" in data.columns and "hotel_rate" in data.columns:
    data["holiday_x_hotel"] = data["holiday_count"] * data["hotel_rate"]

if "exchange_rate" in data.columns and "gdp" in data.columns:
    data["exchange_x_gdp"] = data["exchange_rate"] * data["gdp"]

if "traffic_volume" in data.columns and "aircraft_passengers" in data.columns:
    data["traffic_x_aircraft"] = data["traffic_volume"] * data["aircraft_passengers"]

interaction_cols = [col for col in data.columns if "_x_" in col]
print("Interaction columns:", interaction_cols)

data.head(12)

Interaction columns: ['exchange_x_gdp', 'traffic_x_aircraft']


,country,month,visitor_arrivals,gdp,exchange_rate,public_holiday_count,aircraft_passengers,traffic_volume,hotel_rate,hotel_occupancy,...,traffic_volume_lag_1,traffic_volume_lag_3,traffic_volume_rollmean_3,traffic_volume_rollmean_6,hotel_rate_lag_1,hotel_rate_lag_3,hotel_rate_rollmean_3,hotel_rate_rollmean_6,exchange_x_gdp,traffic_x_aircraft
0,Australia,1978-01-01,20379.0,135608.0,NaN,2,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,1978-02-01,18852.0,135608.0,NaN,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Australia,1978-03-01,20819.0,135608.0,NaN,9,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Australia,1978-04-01,18697.0,135608.0,NaN,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Australia,1978-05-01,19797.0,135608.0,NaN,3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Australia,1978-06-01,18570.0,135608.0,NaN,2,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Australia,1978-07-01,18054.0,135608.0,NaN,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Australia,1978-08-01,19127.0,135608.0,NaN,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Australia,1978-09-01,18587.0,135608.0,NaN,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Australia,1978-10-01,18185.0,135608.0,NaN,2,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
results = []

features = [
    col for col in data.columns
    if col not in ["log_visitor_arrivals", "month", "country"]
]

for country_name, country_df in data.groupby("country"):
    for col in features:
        df_temp = country_df[[col, "log_visitor_arrivals"]].copy()

        # replace inf / -inf with NaN first
        df_temp = df_temp.replace([np.inf, -np.inf], np.nan).dropna()

        # keep only numeric columns
        if not pd.api.types.is_numeric_dtype(df_temp[col]):
            continue

        # skip if too few rows
        if len(df_temp) < 10:
            continue

        X = df_temp[[col]]
        y = df_temp["log_visitor_arrivals"]

        try:
            model = LinearRegression().fit(X, y)
            r2 = model.score(X, y)
            results.append((country_name, col, r2, len(df_temp)))
        except Exception as e:
            print(f"Skipping {country_name} - {col}: {e}")

r2_df = pd.DataFrame(results, columns=["country", "feature", "R2", "n_rows"])
r2_df = r2_df.sort_values(["country", "R2"], ascending=[True, False]).reset_index(drop=True)

r2_df

r2_df.to_csv("../data/important_features.csv", index=False)

In [35]:
results = []

features = [
    col for col in total_data.columns
    if col not in ["log_visitor_arrivals", "month"]
]

for col in features:
    total_df_temp = total_data[[col, "log_visitor_arrivals"]].copy()

    # replace inf / -inf with NaN first
    total_df_temp = total_df_temp.replace([np.inf, -np.inf], np.nan).dropna()

    # keep only numeric columns
    if not pd.api.types.is_numeric_dtype(total_df_temp[col]):
        continue

    # skip if too few rows
    if len(total_df_temp) < 10:
        continue

    X = total_df_temp[[col]]
    y = total_df_temp["log_visitor_arrivals"]

    try:
        model = LinearRegression().fit(X, y)
        r2 = model.score(X, y)
        results.append((col, r2, len(total_df_temp)))
    except Exception as e:
        print(f"Skipping {col}: {e}")

total_r2_df = pd.DataFrame(results, columns=["feature", "R2", "n_rows"])
total_r2_df = total_r2_df.sort_values("R2", ascending=False).reset_index(drop=True)

total_r2_df
total_r2_df.to_csv("../data/total_important_features.csv", index=False)

In [36]:
data.to_csv("../data/feature_engineered_data.csv", index=False)
total_data.to_csv("../data/total_feature_engineered_data.csv", index=False)